# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import matplotlib.pyplot as plt

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()

print(f"{metadata['name']}: {metadata['description']}")
print(f"Schema version: {metadata.get('version', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

**Note:** All entities are referenced by their `@id` fields, as required.

In [ ]:
# Examine record sets, fields, and columns by @id

record_sets = dataset.metadata.record_sets
record_set_ids = [rs['@id'] for rs in record_sets]

print('Record Sets found:')
for rs in record_sets:
    print(f"@id: {rs['@id']} | name: {rs.get('name','')}")

# Now, for each record set, display its fields
for rs in record_sets:
    print(f"\nFields in record set {rs['@id']}:")
    for field in rs['field']:
        print(f"  @id: {field['@id']} | name: {field.get('name','')} | dataType: {field.get('dataType','')}")
        if 'column' in field:
            for col in field['column']:
                print(f"    Column @id: {col['@id']} | name: {col.get('name','')} | dataType: {col.get('dataType','')}")

## 3. Data Extraction
Load data from a record set into a DataFrame for analysis. All references use `@id` fields.

Below, we extract data from each record set and preview column names and sample records.

In [ ]:
# Extract data from each record set using their @ids
dataframes = {}

for record_set in record_set_ids:
    records = list(dataset.records(record_set=record_set))
    df = pd.DataFrame(records)
    dataframes[record_set] = df
    print(f"\nColumns in DataFrame for record set {record_set}: {df.columns.tolist()}")
    print(df.head())

# For demonstration, select the first record set
main_record_set_id = record_set_ids[0]
main_df = dataframes[main_record_set_id]

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on criteria, normalizing numeric fields, and categorizing data using their `@id`s.

We demonstrate filtering, normalization, and grouping operations below.

In [ ]:
# Pick a numeric field by @id for analysis (e.g., GAD-7 score)

# Find numeric fields
numeric_field_id = None
for rs in record_sets:
    if rs['@id'] == main_record_set_id:
        for field in rs['field']:
            if field.get('dataType') in ['schema:Integer', 'schema:Float', 'schema:Number']:
                numeric_field_id = field['@id']
                break
# Fallback if not found: try column level
if numeric_field_id is None:
    for rs in record_sets:
        if rs['@id'] == main_record_set_id:
            for field in rs['field']:
                if 'column' in field:
                    for col in field['column']:
                        if col.get('dataType') in ['schema:Integer', 'schema:Float', 'schema:Number']:
                            numeric_field_id = col['@id']
                            break
                        
print(f"Numeric field selected for filtering: {numeric_field_id}")

threshold = 10

# Filter DataFrame records
if numeric_field_id in main_df.columns:
    filtered_df = main_df[main_df[numeric_field_id] > threshold]
else:
    filtered_df = main_df
    print("Numeric field not found; skipping filter.")

print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalization
if numeric_field_id in filtered_df.columns:
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Grouping: pick a categorical field by @id (e.g., demographic)
group_field_id = None
for rs in record_sets:
    if rs['@id'] == main_record_set_id:
        for field in rs['field']:
            # Exclude numeric types
            if field.get('dataType') not in ['schema:Integer', 'schema:Float', 'schema:Number']:
                group_field_id = field['@id']
                break
# Fallback to column
if group_field_id is None:
    for rs in record_sets:
        if rs['@id'] == main_record_set_id:
            for field in rs['field']:
                if 'column' in field:
                    for col in field['column']:
                        if col.get('dataType') not in ['schema:Integer', 'schema:Float', 'schema:Number']:
                            group_field_id = col['@id']
                            break

print(f"Group field selected: {group_field_id}")

if group_field_id and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
    print(f"Grouped data by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields using their `@id`s.

In [ ]:
# Visualize the distribution of the numeric field (if present)
plt.figure(figsize=(8, 5))
if numeric_field_id in main_df.columns:
    main_df[numeric_field_id].hist(bins=20)
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.title(f'Distribution of {numeric_field_id}')
    plt.show()

# Visualize grouping (if performed)
if group_field_id and group_field_id in main_df.columns:
    group_counts = main_df[group_field_id].value_counts()
    group_counts.plot(kind='bar')
    plt.xlabel(group_field_id)
    plt.ylabel('Number of records')
    plt.title(f'Records by {group_field_id}')
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

Using the `mlcroissant` library, we loaded, reviewed, and analyzed survey data from Kilifi County, Kenya. Record sets, fields, and columns were referenced by their `@id`s. Numeric and categorical analyses demonstrated how to filter, normalize, group, and visualize the dataset for further research. The FAIR² dataset provides valuable insight into mental health metrics, ready for reproducible scientific workflows.